# Gold Validation Notebook — Open Brewery DB

This notebook validates the **Gold layer**.


In [ ]:
%pip install pandas

from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)


## 1) Locate Gold and its matching Silver snapshot

In [ ]:
candidates = sorted(Path("../data/gold/breweries_by_type_location").glob("ingestion_date=*/run_id=*/result.parquet"))
GOLD_FILE = candidates[-1] if candidates else None

GOLD_FILE

In [ ]:
if GOLD_FILE is None or not GOLD_FILE.exists():
    raise FileNotFoundError("No Gold result found. Set GOLD_FILE manually or run the Gold job first.")

gold_df = pd.read_parquet(GOLD_FILE)
gold_df.head(10)


In [ ]:
gold_run_dir = GOLD_FILE.parent
run_id_dir = gold_run_dir.name
ds_dir = gold_run_dir.parent.name

run_id = run_id_dir.split("=", 1)[1]
ds = ds_dir.split("=", 1)[1]

SILVER_ROOT = Path("../data/silver/breweries") / f"ingestion_date={ds}" / f"run_id={run_id}"
SILVER_ROOT, SILVER_ROOT.exists()


In [ ]:
files = list(SILVER_ROOT.iterdir())

silver_files = sorted(SILVER_ROOT.rglob("*.parquet"))
if not silver_files:
    raise FileNotFoundError(f"No Silver parquet files found under {SILVER_ROOT}")

silver_df = pd.concat([pd.read_parquet(f) for f in silver_files], ignore_index=True)
print("Silver rows:", len(silver_df))
print("Gold rows:", len(gold_df))


## 2) Validate Gold schema

In [ ]:
expected_columns = [
    "country",
    "state_province",
    "brewery_type",
    "brewery_count",
    "ingestion_date",
    "run_id",
]

actual_columns = list(gold_df.columns)
pd.DataFrame({
    "expected": expected_columns,
    "present": [c in actual_columns for c in expected_columns],
})


In [ ]:
gold_df.dtypes.astype(str).to_frame("dtype")


## 3) Recompute Gold from Silver and compare

In [ ]:
recomputed = (
    silver_df.groupby(["country", "state_province", "brewery_type"], dropna=False)
    .size()
    .reset_index(name="brewery_count")
    .sort_values(["country", "state_province", "brewery_type"])
    .reset_index(drop=True)
)
recomputed["ingestion_date"] = ds
recomputed["run_id"] = run_id
recomputed = recomputed[expected_columns]

gold_sorted = gold_df.sort_values(["country", "state_province", "brewery_type"]).reset_index(drop=True)

comparison = gold_sorted.merge(
    recomputed,
    on=["country", "state_province", "brewery_type", "ingestion_date", "run_id"],
    how="outer",
    suffixes=("_gold", "_recomputed"),
    indicator=True,
)

comparison.head(20)


In [ ]:
# If this is empty, Gold matches the recomputed aggregation from Silver.
mismatches = comparison[
    (comparison["_merge"] != "both")
    | (comparison["brewery_count_gold"] != comparison["brewery_count_recomputed"])
]
mismatches


## 4) Reconciliation metrics

In [ ]:
metrics = {
    "gold_rows": int(len(gold_df)),
    "gold_total_breweries": int(gold_df["brewery_count"].sum()),
    "silver_rows": int(len(silver_df)),
    "duplicated_gold_groups": int(gold_df.duplicated(subset=["country", "state_province", "brewery_type"]).sum()),
    "negative_counts": int((gold_df["brewery_count"] < 0).sum()),
    "unknown_ratio_gold": round(
        (
            (gold_df["country"] == "unknown")
            | (gold_df["state_province"] == "unknown")
            | (gold_df["brewery_type"] == "unknown")
        ).mean() * 100,
        2,
    ),
}

pd.Series(metrics, name="value").to_frame()


## 5) Distribution analysis

In [ ]:
gold_df.sort_values("brewery_count", ascending=False).head(20)


In [ ]:
gold_df["brewery_type"].astype("string").value_counts(dropna=False).to_frame("groups")


In [ ]:
gold_df.groupby("country", dropna=False)["brewery_count"].sum().sort_values(ascending=False).head(20).to_frame("total_breweries")
